# I²C 통신 · 실습


## 학습 목표

<!-- exercise-guidance:microprocessor/i2c/objective -->
같은 두 Byte도 부호와 센서 범위 설정에 따라 전혀 다른 물리량이 됩니다. 경계값부터 변환을 검증합니다.
<!-- /exercise-guidance:microprocessor/i2c/objective -->


## 실행 조건

<!-- exercise-guidance:microprocessor/i2c/execution -->
장비 없이 byte·raw 기록으로 시작합니다. 실물 센서 확장 시에는 레지스터의 Full-scale 설정과 해당 센서의 감도표를 함께 사용합니다.
<!-- /exercise-guidance:microprocessor/i2c/execution -->


## 두 바이트를 signed 값으로

<!-- exercise-guidance:microprocessor/i2c/activity-1 -->
### 시작 조건

바이트 쌍은 (0x40,0x00),(0xC0,0x00),(0x7F,0xFF),(0x80,0x00),(0xFF,0xFF)입니다. 앞 바이트가 high byte입니다.

### 진행 방법

1. (high<<8)|low로 16-bit unsigned 값을 만드세요.
2. 값이 2^15 이상이면 2^16을 빼 signed 값으로 바꾸세요.
3. 0x8000 경계에서 원본의 부등호를 그대로 사용했을 때와 비교하세요.

### 남길 결과

- 5행 변환 표와 경계값 assert 2개.
<!-- /exercise-guidance:microprocessor/i2c/activity-1 -->


In [ ]:
pairs=[(0x40,0),(0xC0,0),(0x7F,0xFF),(0x80,0),(0xFF,0xFF)]
def signed16(high,low):
    value=(high<<8)|low
    return value-65536 if value>=32768 else value
print([signed16(*p) for p in pairs])
# TODO: unsigned·예상 값과 경계 assert를 추가하세요.


## 결과 기록·자가 점검

<!-- exercise-guidance:microprocessor/i2c/activity-1-record -->
**high:**

**low:**

**unsigned:**

**signed:**

**직접 예상:**

### 확인할 점

- signed 범위 -32768~32767 안에 들어가는가?
- 0x8000은 -32768로 해석되는가?
<!-- /exercise-guidance:microprocessor/i2c/activity-1-record -->


## Raw 값과 물리 단위

<!-- exercise-guidance:microprocessor/i2c/activity-2 -->
### 시작 조건

가속도 raw=[16384,-8192,0], 감도는 16384 LSB/g와 8192 LSB/g 두 조건입니다. Gyro raw=1310에 131 또는 16.4 LSB/(°/s)를 적용합니다.

### 진행 방법

1. 각 감도 조건에서 가속도와 각속도를 계산하세요.
2. 같은 raw 값이라도 센서 설정이 바뀌면 물리량이 달라지는 이유를 적으세요.
3. 원본 GYRO_CONFIG 값과 감도 선택이 맞는지 확인할 항목을 레지스터 비트·감도·단위로 구분하세요.

### 남길 결과

- 가속도·Gyro 변환 표와 설정 확인 항목 3개.
<!-- /exercise-guidance:microprocessor/i2c/activity-2 -->


In [ ]:
acc_raw=[16384,-8192,0]
for scale in [16384.,8192.]:print('acc [g]',[v/scale for v in acc_raw])
print('gyro [deg/s]',1310/131,1310/16.4)
# TODO: 감도·단위 표와 원본 설정 대조 항목을 적으세요.


## 결과 기록·자가 점검

<!-- exercise-guidance:microprocessor/i2c/activity-2-record -->
**raw:**

**설정 감도:**

**변환 값:**

**단위:**

### 확인할 점

- 이미 g 또는 °/s인 라이브러리 출력을 다시 raw처럼 나누지 않았는가?
- MPU6050과 MPU9250을 같은 센서·같은 변환 상수라고 가정하지 않았는가?
<!-- /exercise-guidance:microprocessor/i2c/activity-2-record -->


## 정지 센서 기록의 편차

<!-- exercise-guidance:microprocessor/i2c/activity-3 -->
### 시작 조건

seed=7로 [0,0,16384] LSB에 표준편차 120 LSB의 노이즈를 더한 100개 3축 표본을 생성합니다. 감도는 16384 LSB/g입니다.

### 진행 방법

1. 축별 평균·표준편차와 가속도 벡터의 크기를 계산하세요.
2. 센서 X축에 500 LSB의 상수 bias를 추가한 기록과 비교하세요.
3. offset 추정과 노이즈 제거가 서로 다른 문제임을 설명하세요.

### 남길 결과

- 두 조건 표와 3축 시계열 그림.
<!-- /exercise-guidance:microprocessor/i2c/activity-3 -->


In [ ]:
import numpy as np
rng=np.random.default_rng(7)
raw=np.array([0,0,16384])+rng.normal(0,120,size=(100,3))
biased=raw+np.array([500,0,0])
# TODO: g 변환과 통계·그래프를 작성하세요.


## 결과 기록·자가 점검

<!-- exercise-guidance:microprocessor/i2c/activity-3-record -->
**조건:**

**축별 평균 [g]:**

**축별 표준편차 [g]:**

**평균 벡터 크기 [g]:**

### 확인할 점

- 같은 노이즈 표본에 bias만 추가했는가?
- 평균과 표준편차를 서로 다른 지표로 읽었는가?
<!-- /exercise-guidance:microprocessor/i2c/activity-3-record -->
